In [1]:
import requests

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def get_popular_movies():
    res = requests.get(f"{BASE_URL}/movies")
    return res.json()

def get_movie_details(movie_id):
    res = requests.get(f"{BASE_URL}/movies/{movie_id}")
    return res.json()

def get_movie_credits(movie_id):
    res = requests.get(f"{BASE_URL}/movies/{movie_id}/credits")
    return res.json()

In [2]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "현재 인기 영화 목록을 가져옵니다",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "영화 ID로 영화 상세 정보를 가져옵니다",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "integer",
                        "description": "영화 ID"
                    }
                },
                "required": ["movie_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "영화 출연진 정보를 가져옵니다",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "integer",
                        "description": "영화 ID"
                    }
                },
                "required": ["movie_id"]
            }
        }
    }
]

In [3]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

ModuleNotFoundError: No module named 'dotenv'

In [ ]:
def run_agent(user_input):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "너는 영화 전문가야. 필요하면 함수를 사용해."},
            {"role": "user", "content": user_input}
        ],
        tools=tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    # 함수 호출 발생한 경우
    if message.tool_calls:
        tool_call = message.tool_calls[0]
        function_name = tool_call.function.name
        arguments = eval(tool_call.function.arguments)

        if function_name == "get_popular_movies":
            result = get_popular_movies()

        elif function_name == "get_movie_details":
            result = get_movie_details(arguments["movie_id"])

        elif function_name == "get_movie_credits":
            result = get_movie_credits(arguments["movie_id"])

        # 결과 다시 LLM에 전달
        final_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "user", "content": user_input},
                message,
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(result)
                }
            ]
        )

        return final_response.choices[0].message.content

    else:
        return message.content

In [ ]:
print(run_agent("지금 인기 있는 영화가 뭐야"))
print(run_agent("movie ID 550 영화 알려줘"))
print(run_agent("movie ID 550 출연진 알려줘"))

현재 인기 있는 영화 몇 편을 소개할게요:

1. **Your Heart Will Be Broken**
   - **개봉일**: 2026-03-26
   - **장르**: 드라마, 로맨스
   - **개요**: 고등학생 폴리나는 새로운 학교에서 괴롭힘을 당하다가 주된 괴롭히는 아이인 바르스와 가짜 연애를 시작한다. 이 과정에서 진정한 감정이 싹트지만 가족과 동급생들이 그들을 갈라놓으려 한다.
   - [포스터](https://image.tmdb.org/t/p/w780/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg)

2. **Avatar: Fire and Ash**
   - **개봉일**: 2025-12-17
   - **장르**: SF, 모험, 판타지
   - **개요**: 제이크 설리와 네이티리의 가족이 판도라에서 새로운 위협에 맞서야 하는 이야기.
   - [포스터](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)

3. **Hoppers**
   - **개봉일**: 2026-03-04
   - **장르**: 애니메이션, 가족, SF, 코미디, 모험
   - **개요**: 사람들이 로봇 동물과 교류할 수 있는 기술을 발견하고, 주인공은 이를 통해 동물 세계의 신비를 밝혀낸다.
   - [포스터](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)

4. **Crime 101**
   - **개봉일**: 2026-02-11
   - **장르**: 범죄, 스릴러
   - **개요**: LA의 101 고속도로를 배경으로 한 도둑과 보험 중개인, 그리고 끈질긴 탐정의 이야기가 펼쳐진다.
   - [포스터](https://image.tmdb.org/t/p/w780/tVvpFIoteRHNnoZMhdnwIVwJpCA.jpg)

5. **Scream 7**
   - **개봉일**: 2026-02-25
   - **장르**: 공포

BadRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_EPsvryILe2wU28bUk4jnZo1J", 'type': 'invalid_request_error', 'param': 'messages', 'code': None}}